# ADC Parameter Test
- Default: print 12-bit ADC code every 1s
- Switch 1 pressed: print voltage instead
- Switch 2 pressed: exit

In [2]:
from pop import Potentiometer, Switches, delay

pot = Potentiometer()
sw = Switches()

ADC_BITS = 12
VREF = 3.3
LSB = VREF / (2 ** ADC_BITS)  # 3.3 / 4096
ADC_MAX = (2 ** ADC_BITS) - 1

# False: ADC code mode, True: voltage mode
voltage_mode = False
prev_s1 = True

POLL_MS = 20
PRINT_MS = 1000
elapsed_ms = 0
debounce_ms = 0

print('1s output interval')
print('SW1: toggle mode (ADC <-> Voltage), SW2: exit')

try:
    while True:
        s1 = sw[0].read()  # pressed -> False
        s2 = sw[1].read()  # pressed -> False

        if not s2:
            print('SW2 pressed: program end')
            break

        # SW1 눌렀을 때마다 모드 토글 (디바운스 포함)
        if debounce_ms <= 0 and prev_s1 and (not s1):
            voltage_mode = not voltage_mode
            debounce_ms = 250
            if voltage_mode:
                print('Mode changed: Voltage')
            else:
                print('Mode changed: ADC code')

        prev_s1 = s1

        voltage = pot.readVoltAverage()
        if voltage < 0:
            voltage = 0.0
        elif voltage > VREF:
            voltage = VREF

        adc_code = int(voltage / LSB)
        if adc_code > ADC_MAX:
            adc_code = ADC_MAX

        if elapsed_ms >= PRINT_MS:
            if voltage_mode:
                print(f'Voltage: {voltage:.4f} V')
            else:
                print(f'ADC(12-bit): {adc_code}  bin:{adc_code:012b}')
            elapsed_ms = 0

        delay(POLL_MS)
        elapsed_ms += POLL_MS
        if debounce_ms > 0:
            debounce_ms -= POLL_MS

except KeyboardInterrupt:
    print('KeyboardInterrupt: stopped')


1s output interval
SW1: toggle mode (ADC <-> Voltage), SW2: exit
ADC(12-bit): 28  bin:000000011100
ADC(12-bit): 964  bin:001111000100
ADC(12-bit): 964  bin:001111000100
ADC(12-bit): 964  bin:001111000100
ADC(12-bit): 964  bin:001111000100
ADC(12-bit): 3307  bin:110011101011
ADC(12-bit): 4093  bin:111111111101
Mode changed: Voltage
Voltage: 3.2984 V
Voltage: 1.2281 V
Voltage: 1.2281 V
KeyboardInterrupt: stopped
